# Triple-Barrier Label Selection — AUC method (CPCV path-AUC) vs economic-Sharpe

A **head-to-head** of two ways to *choose* the triple-barrier exit geometry `(pt, sl, h)`, judged by
**learnable, gated out-of-sample Sharpe of the adjusted-signal PnL**.

* **Method A — economic Sharpe (existing study).** Per-instrument geometry from the adjusted-signal
  Sharpe oracle (`triple-barrier-label.ipynb`, CLAUDE.md §11). Loaded from
  `results/triple_barrier_model_geometry.csv`.
* **Method B — AUC method (this notebook).** Geometry chosen by **mean ROC-AUC over the CPCV OOS
  paths** of a fixed **shallow XGB on all features**, over the **same 343-geometry grid**, at three
  scopes: `global` / `per_class` (EQ/EN/ME) / `per_instrument`. CPCV upgrade of
  `src/stml/model/barrier_search.py`.

**PnL convention (the metric that decides the verdict).** The triple-barrier label (or the learnable
`p̂`) only *gates*: `s_adj = s · 1[p̂ ≥ p*]`. The realised strategy return is the **adjusted signal
times the next-bar return**, `s_adj · r_{t+1}` (`r_{t+1} = close_{t+1}/close_t − 1`), accumulated by
`nav_sharpe` (sum of per-event returns; overlapping bets are not compounded). This is the **lag-1
adjusted-signal PnL** Method A was selected on — a single, horizon-invariant return comparable across
geometries. The **realised barrier-touch return** (multi-bar, entry→first touch) is reported only as a
*secondary* reference column, because it spans a different number of bars per `h` and so isn't
comparable across label sets.

**Verdict reported two ways:** the **distribution across the CPCV OOS paths** (dev, 10-block ⇒ 9 paths)
and a single **2022-H1 hold-out** number. The evaluation protocol is identical for all label sets, so
the comparison isolates *which labelling* yields the best learnable adjusted-signal Sharpe.

**Outputs:** `data/triple_barrier_labels_auc.csv` (AUC labels, 3 scopes), `results/triple_barrier_auc_geometry.csv`
(chosen geometry per scope + search path-AUC), `results/triple_barrier_auc_vs_sharpe.csv` (the comparison).

**Caveats (CLAUDE.md §7, §11).** `f2_vol_20` de-annualised by √252 via `events_frame`. `h=1` overlaps
the lag-1 return (circular). Per-instrument CPCV path-AUC is fragile on thin names (`ho1s`, `ng1s`) —
unscorable cells fall back to the global geometry, flagged. Leakage: dev labelling uses
`price_end=VAL_END`; the test partition is opened once via `release_test`.

In [ ]:
import os
# Pin every XGB fit to ONE thread; we parallelise the (heavy) geometry loop across cores instead.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
%matplotlib inline
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
np.random.seed(42)
SEED = 42
N_JOBS = 16                        # process workers for the geometry search (box has 24 cores)
RELOAD_SEARCH = True               # reuse results/triple_barrier_auc_geometry.csv if present (skip the ~40-min search)

from stml.io import _find_repo_root
ROOT = _find_repo_root(Path.cwd().resolve()); RESULTS = ROOT / "results"; DATA = ROOT / "data"
from stml.model.dataset import (load_matrix, close_panel, attach_bar_pos, events_frame,
    select_features, make_xy, align_columns, embargo_map, asset_class_map, scope_iter, DEV_PARTITIONS)
from stml.model.cv import CombinatorialPurgedCV
from stml.model.labels import (triple_barrier_labels, triple_barrier_labels_per_instrument,
    load_instrument_geometry, class_balance, sample_uniqueness)
from stml.model.trees import XGBModel, xgb_baseline_params
from stml.model.optuna_objective import cpcv_oof_auc
from stml.model.evaluate import (release_test, nav_sharpe, bootstrap_returns, decision_threshold,
    per_instrument_breakdown)
from sklearn.metrics import roc_auc_score

matrix = attach_bar_pos(load_matrix(), close_panel())
close_wide = close_panel()
dev_matrix = matrix[matrix.partition.isin(DEV_PARTITIONS)].reset_index(drop=True)
ev_dev = events_frame(dev_matrix)                 # side=f5_signal, sigma=f2_vol_20/sqrt(252)
INSTR = sorted(ev_dev.instrument.unique())
EMB = embargo_map(); ACMAP = asset_class_map()
VAL_END, TEST_PRICE_END = "2021-12-30", "2022-06-30"
GEO_CSV = RESULTS / "triple_barrier_auc_geometry.csv"

PTS = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]; SLS = PTS; HS = [1, 2, 3, 5, 10, 15, 20]
GRID = [(pt, sl, h) for h in HS for pt in PTS for sl in SLS]   # 343, identical to triple-barrier-label.ipynb

SEARCH_BLOCKS, SEARCH_K = 6, 2     # CPCV for the geometry search (15 models / 5 paths)
EVAL_BLOCKS,  EVAL_K    = 10, 2    # CPCV for the 4 finalists (45 models / 9 paths)
MIN_MINORITY = 0.30                # class-balance floor (barrier_search.py)
DROP_REDUNDANT = True              # "all features" = standard pooled set; False -> full 175
FEATS = select_features(dev_matrix, drop_redundant=DROP_REDUNDANT)
print(f"dev events {len(ev_dev)} | instruments {len(INSTR)} | features {len(FEATS)} | grid {len(GRID)}")
print(f"CPCV search ({SEARCH_BLOCKS},{SEARCH_K}) | eval ({EVAL_BLOCKS},{EVAL_K}) | reload_search={RELOAD_SEARCH}")


def lag1_signed_returns(events, close_wide):
    # Per-event next-bar signed return g1 = side * (close_{t+1}/close_t - 1) on each instrument's own
    # calendar -- the tradeable lag-1 PnL of the (adjusted) signal. Geometry-independent.
    out = events[["date", "instrument", "side"]].copy().reset_index(drop=True)
    out["g1"] = np.nan
    for inst, g in out.groupby("instrument"):
        if inst not in close_wide.columns:
            continue
        s = close_wide[inst].dropna(); u = s.pct_change().to_numpy()
        pos = s.index.get_indexer(pd.DatetimeIndex(g["date"])); side = g["side"].to_numpy(float)
        vals = np.full(len(g), np.nan)
        ok = (pos >= 0) & (pos + 1 < len(u))
        vals[ok] = u[pos[ok] + 1] * side[ok]
        out.loc[g.index, "g1"] = vals
    return out[["date", "instrument", "g1"]]


def plateau_pick(table):
    # Robust-plateau pick (cols pt,sl,h,path_auc): rescore each (pt,sl,h) as mean path_auc of itself +
    # its immediate (pt,sl) neighbours at the same h, then argmax (mirrors barrier_search._plateau_scores).
    scored = table[table["path_auc"].notna()].copy()
    if scored.empty:
        return None
    pts = np.sort(scored["pt"].unique()); sls = np.sort(scored["sl"].unique())
    pr = {v: i for i, v in enumerate(pts)}; sr = {v: i for i, v in enumerate(sls)}
    plat = []
    for _, row in scored.iterrows():
        nb = scored[(scored["h"] == row["h"])
                    & (scored["pt"].map(pr).sub(pr[row["pt"]]).abs() <= 1)
                    & (scored["sl"].map(sr).sub(sr[row["sl"]]).abs() <= 1)]
        plat.append(float(nb["path_auc"].mean()))
    scored = scored.assign(plateau=plat)
    return scored.loc[scored["plateau"].idxmax()]


def make_weight_fn(lab_df):
    # Leakage-safe uniqueness weights, recomputed per purged-train fold.
    return lambda tr: sample_uniqueness(lab_df.iloc[tr], close_wide).to_numpy()


def cpcv_paths(model_cls, params, X, y, ret_lag1, ret_real, df, cpcv, p_star, *, seed=SEED, weight_fn=None):
    # Per-CPCV-path OOS AUC + learnable gated Sharpe on BOTH return series: primary = adjusted-signal
    # next-bar return s_adj*r_{t+1} (ret_lag1), secondary = realised barrier-touch return (ret_real).
    # s_adj = s * 1[proba >= p_star]. Mirrors optuna_objective.cpcv_oof_auc's fit/reassemble loop.
    df = df.reset_index(drop=True); X = X.reset_index(drop=True)
    y = np.asarray(y); ret_lag1 = np.asarray(ret_lag1, float); ret_real = np.asarray(ret_real, float)
    n = len(df); dates = pd.to_datetime(df["date"]); blocks = cpcv._date_blocks(dates)
    block_of = np.full(n, -1, dtype=int)
    for b, blk in enumerate(blocks):
        block_of[dates.isin(blk).to_numpy()] = b
    split_pred = {}
    for si, (tr, te) in enumerate(cpcv.split(df)):
        if np.unique(y[tr]).size < 2:
            split_pred[si] = None; continue
        sw = weight_fn(tr) if weight_fn is not None else None
        model = model_cls(params, seed).fit(X.iloc[tr], y[tr], sample_weight=sw)
        pr = np.full(n, np.nan); pr[te] = model.predict_proba(X.iloc[te]); split_pred[si] = pr
    out = []; assignments = cpcv.path_assignments(df)
    for pid in range(cpcv.n_paths()):
        proba = np.full(n, np.nan); complete = True
        for rec in (r for r in assignments if r["path"] == pid):
            pr = split_pred.get(rec["split"])
            if pr is None:
                complete = False; break
            rb = block_of == rec["block"]; proba[rb] = pr[rb]
        if not complete:
            continue
        mask = ~np.isnan(proba)
        if np.unique(y[mask]).size < 2:
            continue
        take = mask & (proba >= p_star)
        rec = {"path": pid, "auc": float(roc_auc_score(y[mask], proba[mask]))}
        for tag, r in (("lag1", ret_lag1), ("real", ret_real)):
            fin = take & np.isfinite(r)
            ns = nav_sharpe(pd.DataFrame({"ret": r[fin]}), np.ones(int(fin.sum()), dtype=bool))
            rec[f"sharpe_{tag}"] = ns["sharpe"]; rec[f"nav_{tag}"] = ns["nav"]; rec[f"n_{tag}"] = ns["n_taken"]
        out.append(rec)
    return out

In [ ]:
# Precompute the triple-barrier labels for all 343 geometries ONCE on the dev events
# (price_end=VAL_END -> no test price touched). Scope-independent, sliced per scope in the search.
t0 = time.time()
LAB = {}
for pt, sl, h in GRID:
    lab = triple_barrier_labels(close_wide, ev_dev, pt=pt, sl=sl, h=h, price_end=VAL_END)
    if not lab.empty:
        LAB[(pt, sl, h)] = lab
print(f"labelled {len(LAB)}/{len(GRID)} geometries on dev in {time.time()-t0:.1f}s")

## AUC geometry search — three scopes (`global` / `per_class` / `per_instrument`)

Per `(scope, cell, geometry)`: merge cached `bin`, screen the **class-balance floor** (`minority ≥ 0.30`),
score by **mean ROC-AUC over the CPCV OOS paths** of the fixed shallow XGB (weightless, faithful to the
reference), pick per cell by the **robust-plateau** rule. Floor-failing / single-class-path geometries are
unscorable (reported in the coverage count). The search ranks *labels*, so it is independent of the PnL
convention used downstream — with `RELOAD_SEARCH=True` it is loaded from the cached CSV.

> Heavy cell: each fit is 1-thread, geometries run in parallel across 16 cores (`loky`).

In [ ]:
def _score_geom(cell_df, lab_bin, pt, sl, h):
    # One floor-passing geometry on one scope cell: mean CPCV OOS path-AUC of the shallow XGB (loky worker).
    dev = cell_df.merge(lab_bin, on=["date", "instrument"], how="inner").reset_index(drop=True)
    X, y = make_xy(dev, FEATS, instrument_dummies=True)
    cpcv = CombinatorialPurgedCV(SEARCH_BLOCKS, SEARCH_K, h=h, embargo_by_instrument=EMB)
    m, s, npaths = cpcv_oof_auc(XGBModel, xgb_baseline_params(), X, y, dev, cpcv, seed=SEED)
    bal = class_balance(dev)
    return {"pt": pt, "sl": sl, "h": h, "n": len(dev), "minority": bal["minority_frac"],
            "pos_rate": bal["pos_rate"], "path_auc": m, "path_auc_std": s, "n_paths": npaths}


def score_cell(cell_df):
    # Pre-screen the floor (cheap, no fit; minority of merged dev == minority of cell-sliced labels),
    # then score only floor-passers in parallel. Floor-failers recorded NaN (never fitted).
    cell_df = cell_df.reset_index(drop=True)
    insts = set(cell_df["instrument"].unique())
    passers, fail = [], []
    for (pt, sl, h), lab in LAB.items():
        lb = lab[lab["instrument"].isin(insts)][["date", "instrument", "bin"]]
        yv = lb["bin"].to_numpy()
        mino = float(min(yv.mean(), 1.0 - yv.mean())) if yv.size else 0.0
        if yv.size > 0 and mino >= MIN_MINORITY:
            passers.append((lb, pt, sl, h))
        else:
            fail.append({"pt": pt, "sl": sl, "h": h, "n": int(yv.size), "minority": mino,
                         "pos_rate": (float(yv.mean()) if yv.size else np.nan),
                         "path_auc": np.nan, "path_auc_std": np.nan, "n_paths": 0})
    ok = (Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(_score_geom)(cell_df, lb, pt, sl, h) for (lb, pt, sl, h) in passers)
          if passers else [])
    return pd.DataFrame(ok + fail)


if RELOAD_SEARCH and GEO_CSV.exists():
    print("RELOAD_SEARCH: using cached geometry", GEO_CSV, "(skipping the search)")
else:
    geo_rows = []
    SCOPE_ARG = {"global": "pooled", "per_class": "per_class", "per_instrument": "per_instrument"}
    for scope in ["global", "per_class", "per_instrument"]:
        t0 = time.time()
        for cell_id, cell_df in scope_iter(dev_matrix, SCOPE_ARG[scope]):
            tc = time.time()
            tbl = score_cell(cell_df); pick = plateau_pick(tbl)
            n_scored = int(tbl["path_auc"].notna().sum())
            if pick is None:
                print(f"  [{scope:14s}] {cell_id:7s}: NO scorable geometry (0/{len(tbl)}) [{time.time()-tc:.0f}s]")
            else:
                print(f"  [{scope:14s}] {cell_id:7s}: pt={pick.pt} sl={pick.sl} h={int(pick.h)} "
                      f"path_auc={pick.path_auc:.3f}  ({n_scored}/{len(tbl)} scorable) [{time.time()-tc:.0f}s]")
            # broadcast the cell pick to its member instruments; thin per-instrument fails -> NaN (fallback later)
            members = (INSTR if scope == "global"
                       else [i for i in INSTR if ACMAP[i] == cell_id] if scope == "per_class"
                       else [cell_id])
            for i in members:
                geo_rows.append({"scope": scope, "instrument": i,
                                 "pt": (float(pick.pt) if pick is not None else np.nan),
                                 "sl": (float(pick.sl) if pick is not None else np.nan),
                                 "h": (int(pick.h) if pick is not None else -1),
                                 "path_auc": (float(pick.path_auc) if pick is not None else np.nan),
                                 "path_auc_std": (float(pick.path_auc_std) if pick is not None else np.nan)})
        print(f"[{scope}] done in {time.time()-t0:.0f}s\n")
    pd.DataFrame(geo_rows).round(4).to_csv(GEO_CSV, index=False)
    print("wrote", GEO_CSV)

In [ ]:
# Build the three AUC geometry maps from the (cached or freshly-written) CSV. Thin per-instrument
# cells that produced no scorable geometry (h=-1 / NaN) fall back to the global pick.
geo_df = pd.read_csv(GEO_CSV)
g_global = geo_df[geo_df.scope == "global"].iloc[0]
G_GLOBAL = (float(g_global.pt), float(g_global.sl), int(g_global.h))

def geom_map(scope):
    sub = geo_df[geo_df.scope == scope].set_index("instrument")
    out = {}
    for i in INSTR:
        if i in sub.index and sub.loc[i, "h"] != -1 and np.isfinite(sub.loc[i, "pt"]):
            out[i] = (float(sub.loc[i, "pt"]), float(sub.loc[i, "sl"]), int(sub.loc[i, "h"]))
        else:
            out[i] = G_GLOBAL   # fallback
    return out

B_global = geom_map("global"); B_class = geom_map("per_class"); B_inst = geom_map("per_instrument")
print("global geometry:", G_GLOBAL)
display(geo_df.round(4))

In [ ]:
# Method A (economic Sharpe, existing) + the three AUC variants. Relabel dev with each geometry and
# attach the lag-1 adjusted-signal return g1 = side * r_{t+1} (geometry-independent), dropping the few
# events without a next bar so the Sharpe series is finite.
A_geom = load_instrument_geometry(RESULTS / "triple_barrier_model_geometry.csv")   # per-instrument, h>=5
LABEL_SETS = {"A_econSharpe": A_geom, "B_global": B_global, "B_class": B_class, "B_inst": B_inst}
G1_DEV = lag1_signed_returns(ev_dev, close_wide)

def dev_frame_for(geom):
    lab = triple_barrier_labels_per_instrument(close_wide, ev_dev, geom, price_end=VAL_END)
    dev = (dev_matrix.merge(lab[["date", "instrument", "t1", "ret", "bin"]],
                            on=["date", "instrument"], how="inner")
                     .merge(G1_DEV, on=["date", "instrument"], how="left"))
    dev = dev[np.isfinite(dev["g1"])].reset_index(drop=True)
    return dev, {i: int(geom[i][2]) for i in geom}

DEVSETS = {name: dev_frame_for(geom) for name, geom in LABEL_SETS.items()}
for name, (dev, _) in DEVSETS.items():
    b = class_balance(dev)
    print(f"{name:14s} n={b['n']:5d} pos_rate={b['pos_rate']:.3f} geom[{INSTR[0]}]={LABEL_SETS[name][INSTR[0]]}")

## Learnable, gated evaluation — adjusted-signal PnL `s_adj · r_{t+1}`

Pooled shallow XGB under CPCV `(10,2)` → **9 OOS backtest paths**. `p* = L/(G+L)` from the dev bootstrap
of the **lag-1 returns** (the series actually traded). Per path: OOS **AUC**, and the **gated Sharpe** of
`s_adj · r_{t+1}` (take where `p̂ ≥ p*`) — the headline. The realised barrier-touch Sharpe is kept as a
secondary reference. Uniqueness weights recomputed per purged-train fold. Boxplot = the lag-1 Sharpe
distribution across the CPCV OOS paths.

In [ ]:
EVAL = {}
for name, (dev, h_by) in DEVSETS.items():
    X, y = make_xy(dev, FEATS, instrument_dummies=True)
    g1 = dev["g1"].to_numpy(float); rr = dev["ret"].to_numpy(float)
    r_g, r_l = bootstrap_returns(pd.DataFrame({"ret": g1}), seed=SEED)   # p* from the traded (lag-1) returns
    p_star = decision_threshold(r_g, r_l)
    cpcv = CombinatorialPurgedCV(EVAL_BLOCKS, EVAL_K, h=max(h_by.values()),
                                 h_by_instrument=h_by, embargo_by_instrument=EMB)
    paths = cpcv_paths(XGBModel, xgb_baseline_params(), X, y, g1, rr, dev, cpcv, p_star,
                       seed=SEED, weight_fn=make_weight_fn(dev))
    pdf = pd.DataFrame(paths)
    EVAL[name] = {"p_star": p_star, "paths": pdf}
    sl1, au = pdf["sharpe_lag1"], pdf["auc"]
    print(f"{name:14s} p*={p_star:.3f} | path-AUC {au.mean():.3f}+/-{au.std():.3f} | "
          f"lag1 gated Sharpe median {sl1.median():.2f} [IQR {sl1.quantile(.25):.2f},{sl1.quantile(.75):.2f}] "
          f"| (realised {pdf['sharpe_real'].median():.2f}) over {len(pdf)} paths")

fig, ax = plt.subplots(figsize=(9, 5))
order = list(EVAL.keys())
ax.boxplot([EVAL[n]["paths"]["sharpe_lag1"].to_numpy() for n in order], labels=order, showmeans=True)
ax.axhline(0, c="k", lw=0.6, ls=":")
ax.set_ylabel("learnable gated Sharpe of s_adj*r_{t+1} (per CPCV OOS path)")
ax.set_title(f"Adjusted-signal Sharpe across {EVAL_BLOCKS}-block CPCV OOS paths -- by label set")
plt.tight_layout(); plt.show()

In [ ]:
# Single test opening (tripwire). Train on full dev, predict 2022-H1, gate at the dev p*; PnL = s_adj*r_{t+1}.
test_matrix = release_test(matrix, final_confirmation=True)
ev_test = events_frame(test_matrix); G1_TEST = lag1_signed_returns(ev_test, close_wide)
print(f"hold-out events {len(ev_test)} ({ev_test.date.min().date()}->{ev_test.date.max().date()})\n")

def gated_sharpe(r, take):
    fin = take & np.isfinite(r)
    return nav_sharpe(pd.DataFrame({"ret": r[fin]}), np.ones(int(fin.sum()), dtype=bool))

HOLD = {}
for name, (dev, h_by) in DEVSETS.items():
    geom = LABEL_SETS[name]
    X_tr, y_tr = make_xy(dev, FEATS, instrument_dummies=True)
    lab_te = triple_barrier_labels_per_instrument(close_wide, ev_test, geom, price_end=TEST_PRICE_END)
    test = (test_matrix.merge(lab_te[["date", "instrument", "ret", "bin"]], on=["date", "instrument"], how="inner")
                       .merge(G1_TEST, on=["date", "instrument"], how="left"))
    test = test[np.isfinite(test["g1"])].reset_index(drop=True)
    X_te, y_te = make_xy(test, FEATS, instrument_dummies=True)
    X_te = align_columns(X_te, list(X_tr.columns))
    model = XGBModel(xgb_baseline_params(), SEED).fit(
        X_tr, y_tr, sample_weight=sample_uniqueness(dev, close_wide).to_numpy())
    proba = model.predict_proba(X_te); p_star = EVAL[name]["p_star"]
    g1 = test["g1"].to_numpy(float); rr = test["ret"].to_numpy(float); take = proba >= p_star
    meta1 = gated_sharpe(g1, take); prim1 = gated_sharpe(g1, np.ones(len(g1), bool)); mreal = gated_sharpe(rr, take)
    auc = float(roc_auc_score(y_te, proba)) if np.unique(y_te).size == 2 else np.nan
    HOLD[name] = {"auc": auc, "n": len(test),
                  "sharpe_lag1": meta1["sharpe"], "nav_lag1": meta1["nav"], "n_taken": meta1["n_taken"],
                  "primary_lag1": prim1["sharpe"], "sharpe_real": mreal["sharpe"],
                  "instruments": test["instrument"].to_numpy(), "y": y_te, "proba": proba}
    print(f"{name:14s} hold-out AUC={auc:.3f} | lag1 gated Sharpe={meta1['sharpe']:.2f} "
          f"(NAV {meta1['nav']:+.3f}, took {meta1['n_taken']}/{len(test)}) | "
          f"lag1 primary-alone={prim1['sharpe']:.2f} | (realised gated {mreal['sharpe']:.2f})")

In [ ]:
rows = []
for name in LABEL_SETS:
    pdf = EVAL[name]["paths"]; h = HOLD[name]; s1 = pdf["sharpe_lag1"]
    rows.append({
        "label_set": name,
        "oof_path_auc": round(float(pdf["auc"].mean()), 4),
        "cpcv_lag1_median": round(float(s1.median()), 3),
        "cpcv_lag1_iqr": round(float(s1.quantile(.75) - s1.quantile(.25)), 3),
        "cpcv_lag1_min": round(float(s1.min()), 3),
        "holdout_lag1_sharpe": round(h["sharpe_lag1"], 3),
        "holdout_lag1_primary": round(h["primary_lag1"], 3),
        "holdout_lag1_nav": round(h["nav_lag1"], 4),
        "holdout_n_taken": int(h["n_taken"]),
        "holdout_auc": round(h["auc"], 4),
        "cpcv_realised_median": round(float(pdf["sharpe_real"].median()), 3),   # secondary reference
        "holdout_realised_sharpe": round(h["sharpe_real"], 3),                  # secondary reference
    })
comp = pd.DataFrame(rows)
comp.to_csv(RESULTS / "triple_barrier_auc_vs_sharpe.csv", index=False)
print("wrote", RESULTS / "triple_barrier_auc_vs_sharpe.csv")
display(comp)

win_dev = comp.loc[comp["cpcv_lag1_median"].idxmax(), "label_set"]
win_ho = comp.loc[comp["holdout_lag1_sharpe"].idxmax(), "label_set"]
print(f"\nVERDICT (adjusted-signal s_adj*r_t+1) -- highest CPCV-path median Sharpe (dev): {win_dev} | "
      f"highest 2022-H1 hold-out Sharpe: {win_ho}")

bw = HOLD[win_ho]
print(f"\nPer-instrument hold-out AUC for {win_ho} (thin names ho1s/ng1s/cl1s surfaced):")
display(per_instrument_breakdown(bw["instruments"], bw["y"], bw["proba"]).round(3))

In [ ]:
# Persist the AUC-method labels (all three scopes) to data/, mirroring data/triple_barrier_labels.csv
# plus a `scope` column. Full event set (all partitions); no price_end truncation (a label may use real
# forward prices -- leakage discipline governs *selection*, which used price_end=VAL_END).
ev_full = events_frame(matrix); part = matrix[["date", "instrument", "partition"]]
frames = []
for scope, geom in [("global", B_global), ("per_class", B_class), ("per_instrument", B_inst)]:
    lab = triple_barrier_labels_per_instrument(close_wide, ev_full, geom)
    lab = lab.merge(ev_full[["date", "instrument", "side", "sigma"]], on=["date", "instrument"], how="left")
    lab = lab.merge(part, on=["date", "instrument"], how="left")
    lab["scope"] = scope
    lab["pt"] = lab["instrument"].map(lambda i: geom[i][0])
    lab["sl"] = lab["instrument"].map(lambda i: geom[i][1])
    lab["h"]  = lab["instrument"].map(lambda i: geom[i][2])
    frames.append(lab)
labels_out = pd.concat(frames, ignore_index=True).rename(columns={"bin": "label"})
labels_out = labels_out[["scope", "instrument", "date", "t1", "partition", "side", "sigma",
                         "pt", "sl", "h", "ret", "label", "touch"]].sort_values(
    ["scope", "instrument", "date"]).reset_index(drop=True)
labels_out["date"] = pd.to_datetime(labels_out["date"]).dt.strftime("%Y-%m-%d")
labels_out["t1"] = pd.to_datetime(labels_out["t1"]).dt.strftime("%Y-%m-%d")
out_csv = DATA / "triple_barrier_labels_auc.csv"
labels_out.to_csv(out_csv, index=False)
print("wrote", out_csv, "| rows", len(labels_out))
display(labels_out.groupby("scope").agg(n=("label", "size"), label1_rate=("label", "mean")).round(3))